# Demo: Discrete Choice and the Logit Model
When a decision-maker picks one option from a finite set, the modeler usually observes only some of the features that drive the choice. A __discrete choice model__ splits utility into an observed part and a random part and returns the probability of each choice. This example builds a multinomial logit model for a commute-mode choice (`car`, `bus`, or `bike`), computes the choice probabilities, tests their sensitivity to a feature change, demonstrates the independence-of-irrelevant-alternatives property, and recovers the probabilities by simulation.

> __Learning Objectives.__
>
> This example builds a logit model and interprets its choice probabilities:
> * __Random utility and the logit model:__ Build the deterministic utility $V_{j}=\boldsymbol{\beta}^{\top}\mathbf{x}_{j}$ from observed features and compute the multinomial logit choice probabilities $P_{j}$.
> * __Feature sensitivity:__ Change a feature (a congestion charge on driving) and show how the choice probabilities shift.
> * __Independence of irrelevant alternatives:__ Add a near-duplicate alternative and show the logit model's IIA property and its limitation.

Let's get started.
___

## Theory
For a decision-maker choosing among $J$ alternatives, the random utility of alternative $j$ is
$$
U_{j} = V_{j} + \varepsilon_{j},
$$
where $V_{j}=\boldsymbol{\beta}^{\top}\mathbf{x}_{j}$ is the __deterministic__ component built from observed features $\mathbf{x}_{j}$ with weights $\boldsymbol{\beta}$, and $\varepsilon_{j}$ is a __random__ component for unobserved features. The decision-maker picks the alternative with the highest utility.

If the $\varepsilon_{j}$ are independent and identically distributed Gumbel (Type-I extreme value), the probability of choosing alternative $j$ has the closed form
$$
P_{j} = \frac{\exp(\mu V_{j})}{\sum_{k=1}^{J}\exp(\mu V_{k})},
$$
the __multinomial logit__ model, where $\mu>0$ is a scale parameter (normalized to $\mu=1$ here), and $P_{j}\in(0,1)$ for each $j$ with $\sum_{j=1}^{J}P_{j}=1$. Two properties follow:
* __Only utility differences matter:__ adding a constant to every $V_{k}$ leaves each $P_{j}$ unchanged.
* __Independence of irrelevant alternatives (IIA):__ the ratio $P_{j}/P_{k}=\exp(\mu(V_{j}-V_{k}))$ depends only on alternatives $j$ and $k$, not on the other alternatives in the set.
___

## Setup
This example uses functions defined in the `src` directory and a small set of external packages. The `include(...)` call below runs `Include.jl`, which activates the local project environment, loads the packages, and includes our code. The first run may take a few minutes while packages are installed and precompiled.

In [ ]:
include("Include.jl");

## The Commute-Mode Choice
A commuter chooses among `car`, `bus`, and `bike`. Each mode has two features: the trip cost (USD) and the travel time (min). We store the features in a matrix `X` with one row per alternative and one column per feature. Both features reduce utility, so the preference weights `β` are negative. We build a `MyLinearRandomUtilityModel` that holds `β` and the logit scale `μ`.

In [ ]:
modes = ["car", "bus", "bike"];

#      cost(USD)  time(min)
X = [   5.0        20.0;    # car
        2.0        40.0;    # bus
        0.0        35.0 ];  # bike

β = [-0.20, -0.05];   # weights on (cost, time); both negative

model = build(MyLinearRandomUtilityModel, (β = β, μ = 1.0));

## Choice Probabilities
We compute the deterministic utility $V_{j}=\boldsymbol{\beta}^{\top}\mathbf{x}_{j}$ for each mode, then the logit choice probabilities, which we store in `P::Vector{Float64}` for use in later cells.

In [ ]:
P = let
    V = deterministic_utility(model, X);
    P = logit_choice_probabilities(model, V);

    for i ∈ eachindex(modes)
        println("$(rpad(modes[i], 5)) V = $(round(V[i], digits = 3))   P = $(round(100*P[i], digits = 1)) %");
    end

    P
end

bar(modes, P, label = "", c = colors[1], bg = "floralwhite",
    background_color_outside = "white", framestyle = :box, ylabel = "Choice probability P")

## Sensitivity: A Congestion Charge
Suppose the city adds a \$10 congestion charge to driving, raising the car's cost from \$5 to \$15. We recompute the probabilities; the baseline shares are shown as points over the new bars.

In [ ]:
let
    X_charge = copy(X);
    X_charge[1,1] += 10.0;   # add \$10 to the car's cost

    V_charge = deterministic_utility(model, X_charge);
    P_charge = logit_choice_probabilities(model, V_charge);

    for i ∈ eachindex(modes)
        println("$(rpad(modes[i], 5)) P: $(round(100*P[i], digits = 1))% -> $(round(100*P_charge[i], digits = 1))%");
    end

    bar(modes, P_charge, label = "with charge", c = colors[3], bg = "floralwhite",
        background_color_outside = "white", framestyle = :box, fg_legend = :transparent,
        ylabel = "Choice probability P");
    scatter!(modes, P, label = "baseline", c = colors[1], ms = 7, mec = colors[1])
    current()
end

## Independence of Irrelevant Alternatives
The logit model has a strong property: the ratio of any two choice probabilities does not depend on the other alternatives. We test it with the classic "red bus / blue bus" thought experiment. Suppose an `express bus` is added that is identical to the existing bus (same cost and time). Intuitively, the express bus should mostly split the current bus riders, leaving `car` and `bike` roughly unchanged. We check what the logit model predicts.

In [ ]:
let
    modes_iia = ["car", "bus", "bike", "express bus"];

    X_iia = [ 5.0  20.0;    # car
              2.0  40.0;    # bus
              0.0  35.0;    # bike
              2.0  40.0 ];  # express bus (identical to bus)

    V_iia = deterministic_utility(model, X_iia);
    P_iia = logit_choice_probabilities(model, V_iia);

    for i ∈ eachindex(modes_iia)
        println("$(rpad(modes_iia[i], 12)) P = $(round(100*P_iia[i], digits = 1)) %");
    end

    println();
    println("car/bike ratio:  before = $(round(P[1]/P[3], digits = 4))   after = $(round(P_iia[1]/P_iia[3], digits = 4))");
    println("bus share:       before = $(round(100*P[2], digits = 1))%   after (bus + express) = $(round(100*(P_iia[2] + P_iia[4]), digits = 1))%");

    bar(modes_iia, P_iia, label = "", c = colors[1], bg = "floralwhite",
        background_color_outside = "white", framestyle = :box, ylabel = "Choice probability P")
    current()
end

The `car`/`bike` ratio is unchanged after adding the express bus; that is IIA. But the combined bus share rises well above the original bus share, because the logit model treats the express bus as a genuinely new option and draws share from `car` and `bike` in proportion to their existing probabilities. A realistic model would let the two near-identical buses compete mostly with each other and leave `car` and `bike` almost unchanged. This limitation is why the nested logit and mixed logit models were developed.

## Simulate Choices
The choice probabilities define a categorical distribution over the alternatives. We draw `N` choices from the baseline probabilities `P` and confirm the empirical shares match.

In [ ]:
let
    Random.seed!(42);
    N = 100_000;
    simulated_shares = simulate_choices(P, N);

    for i ∈ eachindex(modes)
        println("$(rpad(modes[i], 5)) model P = $(round(100*P[i], digits = 1))%   simulated = $(round(100*simulated_shares[i], digits = 1))%");
    end
end;

## Summary
This example built a multinomial logit model for a commute-mode choice from observed features, computed the choice probabilities, tested their sensitivity to a congestion charge, demonstrated the independence-of-irrelevant-alternatives property, and recovered the probabilities by simulation.

> __Key takeaways:__
>
> 1. **Random utility gives choice probabilities:** Splitting utility into an observed part $V_{j}=\boldsymbol{\beta}^{\top}\mathbf{x}_{j}$ and an IID Gumbel random part yields the multinomial logit $P_{j}=\exp(\mu V_{j})/\sum_{k}\exp(\mu V_{k})$.
> 2. **Features move shares:** Changing a feature (a congestion charge on driving) shifts the choice probabilities toward the cheaper alternatives.
> 3. **IIA is a strong assumption:** The logit ratio $P_{j}/P_{k}$ depends only on $j$ and $k$; adding a near-duplicate alternative preserves that ratio but reallocates share implausibly, which is why nested and mixed logit models exist.

This demo assumes the weights $\boldsymbol{\beta}$ and computes probabilities; the graded example estimates $\boldsymbol{\beta}$ from observed choices and reports how the probabilities respond to features. The same random-utility logic underlies the structural discrete-choice view of Markov decision processes later in the course.
___

### Additional Resources
* McFadden, D. (1974). Conditional logit analysis of qualitative choice behavior. In P. Zarembka (Ed.), _Frontiers in Econometrics_ (pp. 105–142). Academic Press.
* Train, K. (2009). _Discrete Choice Methods with Simulation_ (2nd ed.). Cambridge University Press.
* Ben-Akiva, M., & Lerman, S. R. (1985). _Discrete Choice Analysis: Theory and Application to Travel Demand_. MIT Press.